In [ ]:
# For task1b

### SHAP for baseline model

In [ ]:
# read pickle file
import pandas as pd

df8918 = pd.read_pickle('*******.pkl')
df8918


In [ ]:
df8918['event_clinical_yesno'].value_counts()

In [ ]:
df8918_exacerbated_0_1 = df8918[df8918['event_clinical_yesno'].isin([0, 1])]
df8918_exacerbated_0_1

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
from torch.nn.utils import clip_grad_norm_
import wandb   
import os
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

df_copd_asthma_speech = df8918_exacerbated_0_1.copy()

features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['event_clinical_yesno'].values.astype(np.int64)
speaker_ids = df_copd_asthma_speech['sid'].values

In [ ]:
le_speaker = LabelEncoder()
speaker_labels = le_speaker.fit_transform(speaker_ids).astype(np.int64)
num_speakers = len(le_speaker.classes_)



unique_speakers = np.unique(speaker_labels)
speaker_to_idx = {spk: [] for spk in unique_speakers}
for i, spk in enumerate(speaker_labels):
    speaker_to_idx[spk].append(i)

# Step 1: Set aside a test set
train_val_speakers, test_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=42)
# Step 2: Split the remaining 80% into a training set and a validation set.
train_speakers, val_speakers = train_test_split(train_val_speakers, test_size=0.2, random_state=42)

# Get index
train_indices = [i for spk in train_speakers for i in speaker_to_idx[spk]]
val_indices = [i for spk in val_speakers for i in speaker_to_idx[spk]]
test_indices = [i for spk in test_speakers for i in speaker_to_idx[spk]]

X_train, y_emo_train, y_spk_train = features[train_indices], emotion_labels[train_indices], speaker_labels[train_indices]
X_val, y_emo_val, y_spk_val = features[val_indices], emotion_labels[val_indices], speaker_labels [val_indices]
X_test, y_emo_test, y_spk_test = features[test_indices], emotion_labels[test_indices], speaker_labels[test_indices]


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


class MultiTaskDataset(Dataset):
    def __init__(self, features, emotion_labels, speaker_labels):
        self.features = torch.tensor(features)
        self.emotion_labels = torch.tensor(emotion_labels)
        self.speaker_labels = torch.tensor(speaker_labels)
    def __len__(self): return len(self.emotion_labels)
    def __getitem__(self, idx): return self.features[idx], self.emotion_labels[idx], self.speaker_labels[idx]

train_loader = DataLoader(MultiTaskDataset(X_train, y_emo_train, y_spk_train), batch_size=32, shuffle=True)
val_loader = DataLoader(MultiTaskDataset(X_val, y_emo_val, y_spk_val), batch_size=32, shuffle=False)
test_loader = DataLoader(MultiTaskDataset(X_test, y_emo_test, y_spk_test), batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# GRL
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_param):
        ctx.lambda_param = lambda_param
        return x.clone()
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_param * grad_output, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_param=1.0):
        super().__init__()
        self.lambda_param = lambda_param
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_param)

In [ ]:
def compute_j_ratio(embeddings, labels):
    classes = np.unique(labels)
    if len(classes) < 2: return 0.0
    D = embeddings.shape[1]
    overall_mean = np.mean(embeddings, axis=0)
    S_B, S_W = np.zeros((D, D)), np.zeros((D, D))
    for c in classes:
        X_c = embeddings[labels == c]
        mean_c = np.mean(X_c, axis=0)
        n_c = X_c.shape[0]
        diff = (mean_c - overall_mean).reshape(-1, 1)
        S_B += n_c * diff @ diff.T
        cov = np.cov(X_c, rowvar=False)
        S_W += (n_c - 1) * cov
    return np.trace(S_B) / (np.trace(S_W) + 1e-8)

def extract_embeddings(model, loader, device):
    model.eval()
    embs, spks = [], []
    with torch.no_grad():
        for feats, _, spk in loader:
            feats = feats.to(device)
            emb = model(feats)
            embs.append(emb.cpu().numpy())
            spks.append(spk.numpy())
    return np.concatenate(embs, 0), np.concatenate(spks, 0)

In [ ]:
# Model Definition (Shared)
class LocallyEnhancedFFN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=1)
        self.depthwise = nn.Conv1d(64, 64, kernel_size=1, groups=64)
        self.conv2 = nn.Conv1d(64, input_dim, kernel_size=1)
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        residual = x
        x = x.permute(0, 2, 1)
        x = self.conv1(x)
        x = self.depthwise(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.conv2(x)
        x = F.gelu(x)
        x = x.permute(0, 2, 1)
        return x + residual

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=0.5)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = LocallyEnhancedFFN(embed_dim)
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        residual = x
        x_attn = x.permute(1, 0, 2)
        attn_output, _ = self.self_attn(x_attn, x_attn, x_attn)
        attn_output = attn_output.permute(1, 0, 2)
        attn_output = self.dropout(attn_output)
        x = self.norm1(residual + attn_output)
        ffn_output = self.ffn(x)
        return self.norm2(x + ffn_output)

class UpstreamModel(nn.Module):
    def __init__(self, input_dim=54, hidden_dim=128, embed_dim=64):
        super().__init__()
        self.reshape = lambda x: x.unsqueeze(1)
        self.encoder1 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
        self.encoder2 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim//2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        x = self.reshape(x)
        x = self.encoder1(x)
        x = self.encoder2(x)
        x, _ = self.lstm(x)
        x = x.squeeze(1)
        x = self.fc(x)
        x = self.dropout(x)
        return x

class EmotionClassifier(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64,2)
        )
    def forward(self, x):
        return self.classifier(x)

class SpeakerClassifier(nn.Module):
    def __init__(self, embed_dim, num_speakers):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, num_speakers)
        )
    def forward(self, x):
        return self.classifier(x)  # No longer routed through GRL
    
def evaluate_emotion(upstream, cls, loader):
    upstream.eval(); cls.eval()
    probs, labels = [], []
    with torch.no_grad():
        for feats, emo, _ in loader:
            feats = feats.to(device)
            out = cls(upstream(feats))
            prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            probs.append(prob); labels.append(emo.numpy())
    probs = np.concatenate(probs); labels = np.concatenate(labels)
    acc = accuracy_score(labels, probs > 0.5)
    auc = roc_auc_score(labels, probs)
    return acc, auc

def evaluate_speaker_leakage(upstream, loader):
    upstream.eval()
    from sklearn.linear_model import LogisticRegression
    emb_val, spk_val = extract_embeddings(upstream, loader, device)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(emb_val, spk_val)
    pred = clf.predict(emb_val)
    return accuracy_score(spk_val, pred)

In [ ]:
num_speakers = df_copd_asthma_speech['sid'].nunique()
num_speakers

In [ ]:
import shap

# Load device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Create a combined model class (for SHAP analysis)
class FullEmotionModel(nn.Module):
    def __init__(self, upstream, emotion_cls):
        super(FullEmotionModel, self).__init__()
        self.upstream = upstream
        self.emotion_cls = emotion_cls

    def forward(self, x):
        embeddings = self.upstream(x)
        output = self.emotion_cls(embeddings)
        return output

# Load the model and scaler
scaler = joblib.load("****.pkl")
model_state = torch.load("****.pth", map_location=device)

# Initialize model
upstream = UpstreamModel().to(device)
emotion_cls = EmotionClassifier().to(device)

# Load weights
upstream.load_state_dict(model_state['upstream'])
emotion_cls.load_state_dict(model_state['emotion_cls'])

# Create a composite model
full_model = FullEmotionModel(upstream, emotion_cls).to(device)
full_model.eval()  

# Preparing for SHAP Analysis
background_data = X_train  
background_data = scaler.transform(background_data) 
background_tensor = torch.tensor(background_data, dtype=torch.float32).to(device)

# Select samples to explain
sample_to_explain = X_test
sample_to_explain = scaler.transform(sample_to_explain)  
sample_tensor = torch.tensor(sample_to_explain, dtype=torch.float32).to(device)

# Use KernelSHAP
def model_predict(x):
    # Ensure the input is a torch tensor.
    if not isinstance(x, torch.Tensor):
        x = torch.tensor(x, dtype=torch.float32).to(device)

    with torch.no_grad():
        output = full_model(x)
        return torch.softmax(output, dim=1).cpu().numpy()

# creat KernelExplainer
explainer = shap.KernelExplainer(
    model_predict,
    shap.sample(background_data,) 
)

# Calculate SHAP values ​​(computing the contribution for the positive class only)
# For binary classification problems, we are typically concerned with the contribution of the positive class (exacerbation events).
shap_values = explainer.shap_values(
    sample_to_explain,
)

# Obtain the SHAP values ​​for the positive class (index 1)
if isinstance(shap_values, list):
    positive_class_shap = shap_values[1]
else:
    positive_class_shap = shap_values

# Feature Name (named according to my 54 features)
feature_names = df_copd_asthma_speech.iloc[:, :54].columns.tolist() 

In [ ]:
np.save("**********_baseline_shap_values_TASK1_response.npy", shap_values)

In [ ]:
shap_values[1].shape

In [ ]:
X_test.shape

### SHAP for disentanglement model

In [ ]:
# read pickle file
import pandas as pd

df8918 = pd.read_pickle('*******.pkl')
df8918


In [ ]:
df8918['event_clinical_yesno'].value_counts()

In [ ]:
df8918_exacerbated_0_1 = df8918[df8918['event_clinical_yesno'].isin([0, 1])]
df8918_exacerbated_0_1

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
from torch.nn.utils import clip_grad_norm_
import wandb  
import os
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

df_copd_asthma_speech = df8918_exacerbated_0_1.copy()


features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['event_clinical_yesno'].values.astype(np.int64)
speaker_ids = df_copd_asthma_speech['sid'].values

In [ ]:
le_speaker = LabelEncoder()
speaker_labels = le_speaker.fit_transform(speaker_ids).astype(np.int64)
num_speakers = len(le_speaker.classes_)



unique_speakers = np.unique(speaker_labels)
speaker_to_idx = {spk: [] for spk in unique_speakers}
for i, spk in enumerate(speaker_labels):
    speaker_to_idx[spk].append(i)

# Step 1: Set aside a test set
train_val_speakers, test_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=42)
# Step 2: Split the remaining 80% into a training set and a validation set.
train_speakers, val_speakers = train_test_split(train_val_speakers, test_size=0.2, random_state=42)

# Get index
train_indices = [i for spk in train_speakers for i in speaker_to_idx[spk]]
val_indices = [i for spk in val_speakers for i in speaker_to_idx[spk]]
test_indices = [i for spk in test_speakers for i in speaker_to_idx[spk]]

X_train, y_emo_train, y_spk_train = features[train_indices], emotion_labels[train_indices], speaker_labels[train_indices]
X_val, y_emo_val, y_spk_val = features[val_indices], emotion_labels[val_indices], speaker_labels [val_indices]
X_test, y_emo_test, y_spk_test = features[test_indices], emotion_labels[test_indices], speaker_labels[test_indices]


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


class MultiTaskDataset(Dataset):
    def __init__(self, features, emotion_labels, speaker_labels):
        self.features = torch.tensor(features)
        self.emotion_labels = torch.tensor(emotion_labels)
        self.speaker_labels = torch.tensor(speaker_labels)
    def __len__(self): return len(self.emotion_labels)
    def __getitem__(self, idx): return self.features[idx], self.emotion_labels[idx], self.speaker_labels[idx]

train_loader = DataLoader(MultiTaskDataset(X_train, y_emo_train, y_spk_train), batch_size=32, shuffle=True)
val_loader = DataLoader(MultiTaskDataset(X_val, y_emo_val, y_spk_val), batch_size=32, shuffle=False)
test_loader = DataLoader(MultiTaskDataset(X_test, y_emo_test, y_spk_test), batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# GRL
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_param):
        ctx.lambda_param = lambda_param
        return x.clone()
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_param * grad_output, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_param=1.0):
        super().__init__()
        self.lambda_param = lambda_param
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_param)

In [ ]:
def compute_j_ratio(embeddings, labels):
    classes = np.unique(labels)
    if len(classes) < 2: return 0.0
    D = embeddings.shape[1]
    overall_mean = np.mean(embeddings, axis=0)
    S_B, S_W = np.zeros((D, D)), np.zeros((D, D))
    for c in classes:
        X_c = embeddings[labels == c]
        mean_c = np.mean(X_c, axis=0)
        n_c = X_c.shape[0]
        diff = (mean_c - overall_mean).reshape(-1, 1)
        S_B += n_c * diff @ diff.T
        cov = np.cov(X_c, rowvar=False)
        S_W += (n_c - 1) * cov
    return np.trace(S_B) / (np.trace(S_W) + 1e-8)

def extract_embeddings(model, loader, device):
    model.eval()
    embs, spks = [], []
    with torch.no_grad():
        for feats, _, spk in loader:
            feats = feats.to(device)
            emb = model(feats)
            embs.append(emb.cpu().numpy())
            spks.append(spk.numpy())
    return np.concatenate(embs, 0), np.concatenate(spks, 0)

In [ ]:
# Model Definition (Shared)
class LocallyEnhancedFFN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=1)
        self.depthwise = nn.Conv1d(64, 64, kernel_size=1, groups=64)
        self.conv2 = nn.Conv1d(64, input_dim, kernel_size=1)
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        residual = x
        x = x.permute(0, 2, 1)
        x = self.conv1(x)
        x = self.depthwise(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.conv2(x)
        x = F.gelu(x)
        x = x.permute(0, 2, 1)
        return x + residual

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=0.5)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = LocallyEnhancedFFN(embed_dim)
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        residual = x
        x_attn = x.permute(1, 0, 2)
        attn_output, _ = self.self_attn(x_attn, x_attn, x_attn)
        attn_output = attn_output.permute(1, 0, 2)
        attn_output = self.dropout(attn_output)
        x = self.norm1(residual + attn_output)
        ffn_output = self.ffn(x)
        return self.norm2(x + ffn_output)

class UpstreamModel(nn.Module):
    def __init__(self, input_dim=54, hidden_dim=128, embed_dim=64):
        super().__init__()
        self.reshape = lambda x: x.unsqueeze(1)
        self.encoder1 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
        self.encoder2 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim//2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        x = self.reshape(x)
        x = self.encoder1(x)
        x = self.encoder2(x)
        x, _ = self.lstm(x)
        x = x.squeeze(1)
        x = self.fc(x)
        x = self.dropout(x)
        return x

class EmotionClassifier(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64,2)
        )
    def forward(self, x):
        return self.classifier(x)

class SpeakerClassifier(nn.Module):
    def __init__(self, embed_dim, num_speakers):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, num_speakers)
        )
    def forward(self, x):
        return self.classifier(x)  
    
def evaluate_emotion(upstream, cls, loader):
    upstream.eval(); cls.eval()
    probs, labels = [], []
    with torch.no_grad():
        for feats, emo, _ in loader:
            feats = feats.to(device)
            out = cls(upstream(feats))
            prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            probs.append(prob); labels.append(emo.numpy())
    probs = np.concatenate(probs); labels = np.concatenate(labels)
    acc = accuracy_score(labels, probs > 0.5)
    auc = roc_auc_score(labels, probs)
    return acc, auc

def evaluate_speaker_leakage(upstream, loader):
    upstream.eval()
    from sklearn.linear_model import LogisticRegression

    emb_val, spk_val = extract_embeddings(upstream, loader, device)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(emb_val, spk_val)
    pred = clf.predict(emb_val)
    return accuracy_score(spk_val, pred)

In [ ]:
num_speakers = df_copd_asthma_speech['sid'].nunique()
num_speakers

In [ ]:
import shap

# Load device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Create a combined model class (for SHAP analysis)
class FullEmotionModel(nn.Module):
    def __init__(self, upstream, emotion_cls):
        super(FullEmotionModel, self).__init__()
        self.upstream = upstream
        self.emotion_cls = emotion_cls

    def forward(self, x):
        embeddings = self.upstream(x)
        output = self.emotion_cls(embeddings)
        return output

#Load the model and scaler
scaler = joblib.load("*****_lambda0.1.pkl")
model_state = torch.load("*******_lambda0.1.pth", map_location=device)

# Initialize model
upstream = UpstreamModel().to(device)
emotion_cls = EmotionClassifier().to(device)

# Load weights
upstream.load_state_dict(model_state['upstream'])
emotion_cls.load_state_dict(model_state['emotion_cls'])

# Create a composite model
full_model = FullEmotionModel(upstream, emotion_cls).to(device)
full_model.eval()   

# Prepare for SHAP analysis
background_data = X_train  
background_data = scaler.transform(background_data)  
background_tensor = torch.tensor(background_data, dtype=torch.float32).to(device)

# Select the samples to explain
sample_to_explain = X_test
sample_to_explain = scaler.transform(sample_to_explain)   
sample_tensor = torch.tensor(sample_to_explain, dtype=torch.float32).to(device)

# KernelSHAP
def model_predict(x):
    # Ensure the input is a torch tensor.
    if not isinstance(x, torch.Tensor):
        x = torch.tensor(x, dtype=torch.float32).to(device)

    with torch.no_grad():
        output = full_model(x)
        return torch.softmax(output, dim=1).cpu().numpy()

# create KernelExplainer
explainer = shap.KernelExplainer(
    model_predict,
    shap.sample(background_data,
                
                )  
)

# Calculate SHAP values
# For binary classification problems, we are typically concerned with the contribution of the positive class (adverse events).
shap_values = explainer.shap_values(
    sample_to_explain,
  
)

# Obtain the SHAP values ​​for the positive class (index 1)
if isinstance(shap_values, list):
    positive_class_shap = shap_values[1]
else:
    positive_class_shap = shap_values

# Feature Name (named according to my 54 features)
feature_names = df_copd_asthma_speech.iloc[:, :54].columns.tolist()

In [ ]:
shap_values[1].shape

In [ ]:
# save shap values
np.save("******_adv_shap_values_TASK1b_response.npy", shap_values)

### SHAP Plot

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
from torch.nn.utils import clip_grad_norm_
import wandb
import os
import pandas as pd

# Load data
df_copd_asthma_speech = pd.read_pickle('******.pkl')
df_copd_asthma_speech

In [ ]:
df_copd_asthma_speech.columns

In [ ]:
df_copd_asthma_speech.rename(columns={

    'mfcc0' : 'MFCC0',
    'mfcc1' : 'MFCC1',
    'mfcc2' : 'MFCC2',
    'mfcc3' : 'MFCC3',
    'mfcc4' : 'MFCC4',
    'mfcc5' : 'MFCC5',
    'mfcc6' : 'MFCC6',
    'mfcc7' : 'MFCC7',
    'mfcc8' : 'MFCC8',
    'mfcc9' : 'MFCC9',
    'mfcc10' : 'MFCC10',
    'mfcc11' : 'MFCC11',
    'mfcc12' : 'MFCC12',
    'mfcc13' : 'MFCC13',
    'mfcc14' : 'MFCC14',
    'mfcc15' : 'MFCC15',
    'mfcc16' : 'MFCC16',
    'mfcc17' : 'MFCC17',
    'mfcc18' : 'MFCC18',
    'mfcc19' : 'MFCC19',
    'mfcc20' : 'MFCC20',
    'mfcc21' : 'MFCC21',
    'mfcc22' : 'MFCC22',
    'mfcc23' : 'MFCC23',
    'mfcc24' : 'MFCC24',
    'mfcc25' : 'MFCC25',
    'mfcc26' : 'MFCC26',
    'mfcc27' : 'MFCC27',
    'mfcc28' : 'MFCC28',
    'mfcc29' : 'MFCC29',


    'shimmerLocaldB_sma3nz_amean': 'Shimmer_mean',
    'shimmerLocaldB_sma3nz_stddevNorm': 'Shimmer_std',
    'loudness_sma3_amean': 'Loudness_mean',
    'loudness_sma3_stddevNorm': 'Loudness_std',
    'HNRdBACF_sma3nz_amean': 'HNR_mean',
    'HNRdBACF_sma3nz_stddevNorm': 'HNR_std',
    'loudnessPeaksPerSec': 'Rate of loudness peaks',
    'MeanVoicedSegmentLengthSec': 'Length of continuously voice_mean',
    'StddevVoicedSegmentLengthSec': 'Length of continuously voice_std',
    'MeanUnvoicedSegmentLength': 'Number of continuous slience_mean',
    'StddevUnvoicedSegmentLength': 'Number of continuous slience_std',
    'VoicedSegmentsPerSec': 'Number of voice per second',

    'F0semitoneFrom27.5Hz_sma3nz_stddevNorm': 'Pitch_std',
    'F0semitoneFrom27.5Hz_sma3nz_amean': 'Pitch_mean',
    'jitterLocal_sma3nz_amean': 'Jitter_mean',
    'jitterLocal_sma3nz_stddevNorm': 'Jitter_std',
    'F1frequency_sma3nz_amean': 'F1_mean',
    'F1frequency_sma3nz_stddevNorm': 'F1_std',
    'F2frequency_sma3nz_amean': 'F2_mean',
    'F2frequency_sma3nz_stddevNorm': 'F2_std',
    'F3amplitudeLogRelF0_sma3nz_amean': 'F3_mean',
    'F3frequency_sma3nz_stddevNorm': 'F3_std',
    'F1bandwidth_sma3nz_amean': 'F1_bandwidth_mean',
    'F1bandwidth_sma3nz_stddevNorm': 'F1_bandwidth_std',




    
}, inplace=True)

In [ ]:
feature_names = df_copd_asthma_speech.iloc[:, :54].columns.tolist() 
feature_names

In [ ]:
# read .npy file
import numpy as np
base_shap_values = np.load("*****_baseline_shap_values_TASK1_response.npy", allow_pickle=True)
adv_shap_values = np.load("*****_adv_shap_values_TASK1b_response.npy", allow_pickle=True)

In [ ]:
shap_baseline = base_shap_values[1]  # Positive class
shap_adv = adv_shap_values[1]  # Positive class

In [ ]:
shap_baseline.shape

In [ ]:
shap_adv.shape

In [ ]:
import numpy as np

#Calculate the mean |SHAP| for each feature (global importance)
mean_shap_base = np.mean(np.abs(shap_baseline), axis=0)  # (54,)
mean_shap_adv  = np.mean(np.abs(shap_adv),      axis=0)  # (54,)

#Calculate change (absolute difference)
change_abs = mean_shap_adv - mean_shap_base  # Positive: enhancement; negative: inhibition


In [ ]:
epsilon = 1e-8
change_relative = change_abs / (np.abs(mean_shap_base) + epsilon)



# Sort and identify the top_n features with the largest changes (based on absolute values)
top_n = 10
sorted_indices_by_change = np.argsort(np.abs(change_abs))[::-1]  # From largest to smallest
top_change_indices = sorted_indices_by_change[:top_n]

# identify those that are most suppressed and those that are most enhanced
suppressed_indices = np.argsort(change_abs)[:top_n]      # Most negative (inhibited)
enhanced_indices   = np.argsort(change_abs)[-top_n:][::-1]  # The positive (enhanced)

In [ ]:
# Construct the resulting DataFrame
df_all = pd.DataFrame({
    'Feature': feature_names,
    'Mean|SHAP|_Baseline': mean_shap_base,
    'Mean|SHAP|_Adv': mean_shap_adv,
    'Change_Abs': change_abs,
    'Change_Relative': change_relative
}).round(6)

# Sort by the absolute value of the change
df_top_change = df_all.iloc[top_change_indices].copy()
df_suppressed = df_all.iloc[suppressed_indices].copy()
df_enhanced   = df_all.iloc[enhanced_indices].copy()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Set the global font to Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.weight'] = 'bold'  

top_n = 5

# The most suppressed (the smallest)
suppressed_indices = np.argsort(change_abs)[:top_n]  # The 5 smallest

# Most enhanced (greatest)
enhanced_indices = np.argsort(change_abs)[-top_n:]   # The 5 largest

top_indices = np.concatenate([suppressed_indices, enhanced_indices])


# Extract the absolute changes and names of the top features.
top_change_abs = change_abs[top_indices]
top_features = [feature_names[i] for i in top_indices]

# Sort by the absolute change values ​​themselves (from smallest to largest) for the purpose of plotting.
order = np.argsort(top_change_abs)  
top_features_sorted = [top_features[i] for i in order]
top_change_abs_sorted = top_change_abs[order]

# Set colors: red for negative values, blue for positive values
colors = ['red' if x < 0 else 'blue' for x in top_change_abs_sorted]

# plotting
plt.figure(figsize=(10, 6))
y_pos = np.arange(len(top_features_sorted))

bars = plt.barh(
    y_pos,
    top_change_abs_sorted,
    color=colors,
    alpha=0.7,
    edgecolor='black',
    linewidth=0.8
)

# Add value labels to the bars

plt.yticks(y_pos, top_features_sorted)
plt.axvline(0, color='black', linewidth=0.8)

# Invert the y-axis so that the most inhibited items appear at the top.
plt.gca().invert_yaxis()

plt.grid(True, axis='x', alpha=0.8)
plt.tight_layout()

#plt.savefig("*******.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
top_features_sorted

In [ ]:
print(np.array2string(top_change_abs, suppress_small=True, precision=10))
